# Bedrock Narrative Reports

Uses Amazon Bedrock (Claude) to generate natural-language scouting narratives for the top players based on their AWI and PQI scores.

**Prerequisites:** `results/awi_full.csv` and `results/pqi_full.csv` must exist (run `run_awi_pipeline` and `run_pqi_pipeline` first).

**Output:** `results/narratives.csv` , one row per player with a generated narrative string.

In [1]:
import os, sys
from pathlib import Path

# Find project root (CLAUDE.md marker) and chdir to notebooks/
# so ../results/ and ../figures/ resolve correctly from any launch CWD.
_root = next(
    (p for p in [Path().resolve(), *Path().resolve().parents] if (p / "CLAUDE.md").exists()),
    None,
)
if _root is None:
    raise RuntimeError("Cannot locate project root — CLAUDE.md not found.")
os.chdir(_root / "notebooks")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


## Step 1. Initialise Bedrock Client

Creates an authenticated Amazon Bedrock client for the `eu-central-1` region.
Requires a valid AWS session (SSO or environment credentials).

In [2]:
from src.bedrock_client import create_bedrock_client, batch_generate_narratives, generate_match_summary

client = create_bedrock_client(region="eu-central-1")
print("Bedrock client created successfully")

Bedrock client created successfully


## Step 2. Load AWI and PQI Results

Reads the pre-computed AWI (Awareness Index) and PQI (Pressure Quality Index) CSVs.
These are the inputs to the narrative generation step.

In [3]:
import pandas as pd

awi_df = pd.read_csv("../results/awi_full.csv")
pqi_df = pd.read_csv("../results/pqi_full.csv")
print(f"AWI rows: {len(awi_df)}, PQI rows: {len(pqi_df)}")

AWI rows: 400, PQI rows: 400


## Step 3. Generate Narratives via Bedrock

Calls `batch_generate_narratives` which selects the top-N players by AWI, merges their PQI sub-scores, and sends a structured prompt to Claude on Bedrock.

Each narrative summarises a player's scanning behaviour, pressure quality, and positional context in plain English.

In [5]:
narratives_df = batch_generate_narratives(client, awi_df, pqi_df, top_n=10)
print(f"Generated {len(narratives_df)} narratives")
narratives_df.head()

Generated 10 narratives


,jersey,team,match_id,phase_label,narrative
0,6,0,SGE-FCB,1st half,Oscar Winther Höjlund's Awareness Index (AWI) ...
1,16,1,SGE-FCU,1st half,Hugo Emanuel Larsson's Awareness Index of 26.3...
2,6,1,FCU-FCB,1st half,Joshua Kimmich’s Awareness Index (AWI) of 23.4...
3,8,0,SGE-FCU,1st half,Rani Khedira's Awareness Index (AWI) of 22.9 s...
4,6,0,SGE-FCB,2nd half,Oscar Winther Höjlund's Awareness Index (AWI) ...


## Step 4. Save and Preview

Persists the narratives to `results/narratives.csv` and prints the top player's narrative as a quick sanity check.

In [ ]:
narratives_df.to_csv("../results/narratives.csv", index=False)
print("Saved to ../results/narratives.csv")
print("\n--- Top Player Narrative ---")
if len(narratives_df) > 0:
    print(narratives_df.iloc[0]["narrative"])

Saved to ../results/narratives.csv

--- Top Player Narrative ---
Oscar Winther Höjlund's Awareness Index of 26.9 scans per minute, significantly surpassing the league average of 15.6, indicates an exceptional cognitive awareness style. This score, ranking first among 400 players, suggests that Höjlund is highly perceptive on the field, effectively processing spatial information and anticipating opponents' movements. His ability to scan the field frequently allows him to make quick decisions and support teammates, making him a formidable defensive presence.

Höjlund's Pressure Quality Index of 63.9/100 highlights a solid pressing mechanic, particularly in proximity, where he scored 96.0/100. However, his lower orientation (59.0/100) and stance (38.1/100) scores suggest room for improvement in positioning and body posture during pressure. Focusing on enhancing his orientation and stance will improve his overall pressing effectiveness, ensuring he maintains optimal balance and readiness d